# Simpsons Character Classification

This project classifies characters from *The Simpsons* using transfer learning with PyTorch. The goal is to build a practical image-classification pipeline: dataset loading, stratified validation, class-imbalance handling, model fine-tuning, error analysis, and Kaggle-style submission generation.

The repository is structured as a small portfolio project: the notebook explains the experiment end to end, while reusable training, visualization, and label-audit code lives in separate Python modules.


## What this demonstrates

- Transfer learning with a pretrained computer-vision model.
- Clean train/validation split with stratification.
- Handling class imbalance with weighted sampling and weighted loss.
- Reusable project modules instead of notebook-only helper code.
- Error analysis workflow for identifying suspicious labels.
- Reproducible Colab setup with GitHub-hosted code and dataset release assets.


The public test set contains 991 images. The model predicts one character label per image and writes the final predictions to a submission CSV.


## 1. Environment Setup


This section imports the core libraries, verifies GPU availability, and mounts Google Drive for persistent artifacts such as checkpoints, audit tables, and the cached dataset archive.


In [ ]:
# we will verify that GPU is enabled for this notebook
# following should print: CUDA is available!  Training on GPU ...
#
# if it prints otherwise, then you need to enable GPU:
# from Menu > Runtime > Change Runtime Type > Hardware Accelerator > GPU

import torch
import numpy as np

train_on_gpu = torch.cuda.is_available()

if not train_on_gpu:
    print('CUDA is not available.  Training on CPU ...')
else:
    print('CUDA is available!  Training on GPU ...')


from google.colab import drive
drive.mount('/content/drive')


In [ ]:
!nvidia-smi


In [ ]:
import numpy as np
import pandas as pd
import torch.nn as nn
from pathlib import Path
%matplotlib inline


## 2. Project Configuration

The notebook loads source code from GitHub and stores only large runtime artifacts on Google Drive. This keeps the repository lightweight while making Colab runs reproducible.


In [ ]:
# Training device
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Dataset paths after extracting the archive in Colab
TRAIN_DIR = Path("/content/train")
TEST_DIR = Path("/content/testset")
SAMPLE_SUBMISSION_PATH = Path("/content/sample_submission.csv")

# Load project code from GitHub. Use Drive only for large runtime artifacts.
import subprocess
import sys

REPO_URL = "https://github.com/eritry/simpsons-classification.git"
CODE_DIR = Path("/content/simpsons-classification")

if (CODE_DIR / ".git").exists():
    subprocess.run(["git", "-C", str(CODE_DIR), "pull", "--ff-only"], check=True)
elif not (CODE_DIR / "training.py").exists():
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(CODE_DIR)], check=True)

if str(CODE_DIR) not in sys.path:
    sys.path.insert(0, str(CODE_DIR))

# Single Google Drive root for persistent artifacts.
DRIVE_ARTIFACT_DIR = Path("/content/drive/MyDrive/Colab Notebooks/simpsons")
CHECKPOINT_DIR = DRIVE_ARTIFACT_DIR / "checkpoints"
LABEL_AUDIT_DIR = DRIVE_ARTIFACT_DIR / "label_audit"
DATASET_DIR = DRIVE_ARTIFACT_DIR / "dataset"
DATASET_RELEASE_URL = "https://github.com/eritry/simpsons-classification/releases/download/dataset-v1/journey-springfield.zip"
DATASET_ARCHIVE_PATH = DATASET_DIR / "journey-springfield.zip"

BATCH_SIZE = 64

for directory in [DRIVE_ARTIFACT_DIR, CHECKPOINT_DIR, LABEL_AUDIT_DIR, DATASET_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

# CSV with manually reviewed suspicious labels, versioned with the repository.
REVIEW_PATH = CODE_DIR / "artifacts" / "label_audit" / "suspicious_manual.csv"


## 3. Data Loading and Preprocessing


### Dataset Download and Cache

The dataset archive is hosted as a GitHub Release asset. On the first run, the notebook downloads it to Google Drive; later runs reuse the cached archive and only extract it into the current Colab runtime.


In [ ]:
from data_io import ensure_dataset

ensure_dataset(
    archive_url=DATASET_RELEASE_URL,
    archive_path=DATASET_ARCHIVE_PATH,
    extract_dir="/content",
    required_paths=[TRAIN_DIR, TEST_DIR, SAMPLE_SUBMISSION_PATH],
)


In [ ]:
# The dataset is now available under /content/train, /content/testset, and /content/sample_submission.csv.


In [ ]:
# Optional label-fix block based on a manually reviewed CSV.
# By default no files are moved. Review REVIEW_PATH and set APPLY_LABEL_FIXES=True first.
from label_audit import move_reviewed_files_to_class

APPLY_LABEL_FIXES = False


In [ ]:
if APPLY_LABEL_FIXES:
    if not REVIEW_PATH.exists():
        raise FileNotFoundError(f"Manual review file not found: {REVIEW_PATH}")

    review_df = pd.read_csv(REVIEW_PATH)
    moved_paths = move_reviewed_files_to_class(
        review_df=review_df,
        dry_run=False,
        dataset_root=TRAIN_DIR,
    )
else:
    moved_paths = {}
    print("Label fixing is disabled. Set APPLY_LABEL_FIXES = True to enable it.")


In [ ]:
from dataset import collect_image_files

train_val_files, test_files = collect_image_files(TRAIN_DIR, TEST_DIR)
print("Total training/validation images:", len(train_val_files))
print("Test images:", len(test_files))


### Label Encoding

Class names are inferred from parent directory names. `LabelEncoder` maps these text labels to integer class IDs for training and later converts model predictions back to class names for the submission file.


In [ ]:
from dataset import create_label_encoder

label_encoder, train_val_labels = create_label_encoder(train_val_files)


### Train/Validation Split

The original training folder is split into training and validation subsets. The split is stratified by class label so that rare and frequent characters are represented proportionally in both subsets.


In [ ]:
from dataset import split_train_val_files

train_files, val_files = split_train_val_files(
    train_val_files=train_val_files,
    train_val_labels=train_val_labels,
    test_size=0.25,
    random_state=42,
)


### Dataset and DataLoader Objects

`SimpsonsDataset` applies stronger image augmentation during training and deterministic preprocessing for validation/test inference. This mirrors a standard production pattern: augment only the training signal, keep evaluation stable.


In [ ]:
from dataset import SimpsonsDataset, create_datasets

train_dataset, val_dataset = create_datasets(
    train_files=train_files,
    val_files=val_files,
    label_encoder=label_encoder,
)


In [ ]:
print("Train dataset examples:", len(train_dataset))
print("Validation dataset examples:", len(val_dataset))


In [ ]:
batch_size = BATCH_SIZE


### Class Distribution

The dataset is imbalanced: major characters have many more samples than rare characters. The plot below makes that imbalance explicit before training.


In [ ]:
from visualization import plot_train_val_distribution

distribution_df = plot_train_val_distribution(
    train_dataset=train_dataset,
    val_dataset=val_dataset,
    title="Train vs Validation distribution by Simpsons character",
)


### Imbalance Handling

A `WeightedRandomSampler` increases the probability of sampling under-represented classes. I use inverse square-root class frequency rather than full inverse frequency to reduce imbalance without making rare classes dominate every batch.


In [ ]:
from dataset import create_weighted_dataloaders

loaders, class_counts = create_weighted_dataloaders(
    train_dataset=train_dataset,
    val_dataset=val_dataset,
    label_encoder=label_encoder,
    batch_size=batch_size,
    train_num_workers=2,
    val_num_workers=0,
)

train_loader = loaders["train"]
val_loader = loaders["val"]

print("Train size:", len(train_dataset))
print("Val size:", len(val_dataset))
print("Number of classes:", len(label_encoder.classes_))


### Visual Sanity Check

Before training, it is useful to inspect augmented batches manually. This catches obvious preprocessing issues such as wrong labels, broken images, or overly aggressive transforms.


Visualization helpers are implemented in `visualization.py`. Keeping them outside the notebook makes the notebook easier to read and the helpers easier to reuse.


In [ ]:
from visualization import (
    show_images_from_loader,
    show_images_with_predictions,
)


In [ ]:
show_images_from_loader(n_rows=3, n_cols=4, loader=train_loader, label_encoder=label_encoder)


## 4. Model Architecture


### Model Candidates

This project compares three levels of model complexity:

- `SimpleCNN`: a small from-scratch baseline that establishes whether the pipeline learns at all.
- `DenseNet121`: a strong ImageNet-pretrained transfer-learning baseline.
- `EfficientNetV2-S`: the default model used in the main pipeline, also initialized from ImageNet weights.

The notebook trains one selected model at a time to keep Colab runtime manageable. The comparison table below records representative runs for resume/interview discussion.


### Why Compare These Models

The comparison separates data-pipeline quality from architecture strength. If `SimpleCNN` works but underperforms, the pipeline is probably valid and transfer learning is the right next step. Comparing DenseNet121 and EfficientNetV2-S then shows how architecture choice, optimizer settings, and fine-tuning strategy affect macro F1 on an imbalanced dataset.


In [ ]:
# Choose one model for this notebook run.
# Options: "simple_cnn", "densenet121", "efficientnet_v2_s"
MODEL_NAME = "efficientnet_v2_s"

TRANSFER_MODELS = {"densenet121", "efficientnet_v2_s"}


In [ ]:
from model import count_parameters, create_model

selected_model = create_model(
    model_name=MODEL_NAME,
    num_classes=len(label_encoder.classes_),
    dropout=0.3,
).to(DEVICE)

print("Selected model:", MODEL_NAME)
print("Total parameters:", f"{count_parameters(selected_model):,}")
print("Trainable parameters:", f"{count_parameters(selected_model, trainable_only=True):,}")


## 5. Training Utilities

Training, evaluation, checkpointing, plotting, and prediction helpers are imported from `training.py`. The notebook keeps experiment orchestration visible while moving reusable mechanics into focused modules.


In [ ]:
from training import (
    make_training_history,
    plot_history,
    predict,
    train_CNN,
)


In [ ]:
history = make_training_history()


## 6. Training and Validation


### Optimization Strategy

For `SimpleCNN`, all parameters are trained from scratch with Adam. For pretrained transfer-learning models, training uses two stages: first train only the new classifier head, then unfreeze the feature extractor and fine-tune with lower learning rates for pretrained layers.


### Validation metric

The training loop tracks both accuracy and macro F1. Macro F1 is important here because the dataset is imbalanced: it gives rare classes the same weight as frequent classes when summarizing performance.


In [ ]:
# Use weighted CrossEntropyLoss for the fine-tuning stage

class_weights = 1.0 / np.sqrt(np.maximum(class_counts, 1))
class_weights = class_weights / class_weights.mean()

class_weights = torch.tensor(
    class_weights,
    dtype=torch.float32,
    device=DEVICE
)

# The scheduler must be created after the optimizer. Use a factory and call it
# after creating the optimizer for a specific training stage.
def make_cosine_scheduler(optimizer, t_max, eta_min=1e-6):
    return torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer,
        T_max=t_max,
        eta_min=eta_min,
    )


In [ ]:
if MODEL_NAME == "simple_cnn":
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(
        selected_model.parameters(),
        lr=1e-3,
    )

    history = train_CNN(
        model=selected_model,
        num_epochs=11,
        dataloaders=loaders,
        optimizer=optimizer,
        loss_func=criterion,
        device=DEVICE,
        val_every_steps=100,
        save_dir=CHECKPOINT_DIR / MODEL_NAME,
    )
else:
    # Stage 1: train only the new classifier head.
    for param in selected_model.parameters():
        param.requires_grad = False

    for param in selected_model.classifier.parameters():
        param.requires_grad = True

    criterion = nn.CrossEntropyLoss(label_smoothing=0.02)

    optimizer = torch.optim.AdamW(
        selected_model.classifier.parameters(),
        lr=1e-3,
        weight_decay=1e-4,
    )

    history = train_CNN(
        model=selected_model,
        num_epochs=3,
        dataloaders=loaders,
        optimizer=optimizer,
        loss_func=criterion,
        device=DEVICE,
        val_every_steps=100,
        save_dir=CHECKPOINT_DIR / f"{MODEL_NAME}_stage1",
    )

    # Stage 2: unfreeze the backbone and fine-tune with lower learning rates.
    for param in selected_model.parameters():
        param.requires_grad = True

    if MODEL_NAME == "densenet121":
        parameter_groups = [
            {"params": selected_model.features.parameters(), "lr": 1e-4},
            {"params": selected_model.classifier.parameters(), "lr": 1e-3},
        ]
        smoothing = 0.10
    else:
        parameter_groups = [
            {"params": selected_model.features.parameters(), "lr": 1e-5},
            {"params": selected_model.classifier.parameters(), "lr": 3e-4},
        ]
        smoothing = 0.02

    optimizer = torch.optim.AdamW(
        parameter_groups,
        weight_decay=1e-4,
    )

    criterion = nn.CrossEntropyLoss(
        weight=class_weights,
        label_smoothing=smoothing,
    )

    stage2_scheduler = make_cosine_scheduler(optimizer, t_max=10)

    history = train_CNN(
        model=selected_model,
        num_epochs=10,
        dataloaders=loaders,
        optimizer=optimizer,
        loss_func=criterion,
        device=DEVICE,
        val_every_steps=100,
        save_dir=CHECKPOINT_DIR / f"{MODEL_NAME}_stage2",
        history=history,
        scheduler=stage2_scheduler,
    )

plot_history(history)


In [ ]:
from sklearn.metrics import classification_report

selected_model.eval()
all_preds = []
all_targets = []

with torch.no_grad():
    for images, targets in val_loader:
        images = images.to(DEVICE)
        targets = targets.to(DEVICE)

        logits = selected_model(images)
        preds = logits.argmax(dim=1)

        all_preds.extend(preds.cpu().numpy())
        all_targets.extend(targets.cpu().numpy())

print(classification_report(
    all_targets,
    all_preds,
    target_names=label_encoder.classes_,
    digits=4
))


### Model Comparison

Representative results show the role of each model. `SimpleCNN` is a useful baseline but overfits: it reaches high train F1 while validation macro F1 saturates much lower. Pretrained models improve sample efficiency and rare-class performance.

| Model | Pretraining | Parameters | Validation Macro F1 | Kaggle Score | Role |
|---|---:|---:|---:|---:|---|
| SimpleCNN | No | 180,762 | ~0.79 | not used as final | From-scratch baseline |
| DenseNet121 | ImageNet | ~7.2M | ~0.95 | ~0.99256 | Strong transfer-learning reference |
| EfficientNetV2-S | ImageNet | ~21M | ~0.85 | ~0.97236 | Current default pipeline |

DenseNet121 is included as an important comparison point because the reference experiment reached the strongest score with it. EfficientNetV2-S remains the default here because it keeps the notebook focused on a clean, modular transfer-learning workflow; the model selector above makes it easy to rerun the same pipeline with DenseNet121.


In [ ]:
comparison_df = pd.DataFrame([
    {
        "model": "SimpleCNN",
        "pretraining": "None",
        "parameters": "180,762",
        "validation_macro_f1": "~0.79",
        "kaggle_score": "baseline only",
        "role": "from-scratch baseline",
    },
    {
        "model": "DenseNet121",
        "pretraining": "ImageNet",
        "parameters": "~7.2M",
        "validation_macro_f1": "~0.95",
        "kaggle_score": "~0.99256",
        "role": "strong transfer-learning reference",
    },
    {
        "model": "EfficientNetV2-S",
        "pretraining": "ImageNet",
        "parameters": "~21M",
        "validation_macro_f1": "~0.85",
        "kaggle_score": "~0.97236",
        "role": "current default pipeline",
    },
])

comparison_df


## 7. Error Analysis and Label Audit

After training, the model is used to find examples where it disagrees with the current folder label while being highly confident. These cases are good candidates for manual label review. This is not automatic relabeling: the CSV is meant to support human inspection.


In [ ]:
from label_audit import (
    run_label_audit,
    show_review_examples,
)


In [ ]:
review_csv_path = LABEL_AUDIT_DIR / "suspicious_manual_review.csv"
print("Label audit directory:", LABEL_AUDIT_DIR)


In [ ]:
audit_dataset, df_all, df_suspicious = run_label_audit(
    model=selected_model,
    train_files=train_val_files,
    label_encoder=label_encoder,
    dataset_cls=SimpsonsDataset,
    device=DEVICE,
    batch_size=64,
    num_workers=0,
    min_confidence=0.90,
    min_margin=0.20,
    top_n=300,
    save_all_csv_path=LABEL_AUDIT_DIR / "all_predictions.csv",
    save_review_csv_path=LABEL_AUDIT_DIR / "suspicious_labels_review.csv",
)


Label-audit helpers are implemented in `label_audit.py`. They scan predictions, rank suspicious examples, save review CSV files, and visualize candidates for manual inspection.


In [ ]:
review_df = df_suspicious.copy()
review_df["action"] = ""          # options: "move", "keep", "skip"
review_df["correct_label"] = ""   # for example: "homer_simpson"

review_csv_path = LABEL_AUDIT_DIR / "suspicious_manual_new.csv"
review_df.to_csv(review_csv_path, index=False)

show_review_examples(
    review_df=review_df,
    dataset=audit_dataset,
    start_pos=0,
    n_rows=4,
    n_cols=3,
)


### Prediction Confidence Visualization

The next visualization samples validation images and overlays the model's top prediction and confidence. This is a quick qualitative check of calibration and common confusions.


`show_images_with_predictions` is implemented in `visualization.py`.


In [ ]:
show_images_with_predictions(
    n_rows=3,
    n_cols=3,
    dataset=val_dataset,
    model=selected_model,
    label_encoder=label_encoder,
    device=DEVICE,
)


## 8. Kaggle-Style Submission


Create a deterministic test DataLoader. Test samples have no labels, so the dataset returns only transformed images.


In [ ]:
from dataset import create_test_loader

test_dataset, test_loader = create_test_loader(
    test_files=test_files,
    label_encoder=label_encoder,
    batch_size=batch_size,
)


The `predict` helper returns integer class IDs for all test images.


Load the best checkpoint if it exists, then run inference on the test set.


In [ ]:
from submission import load_best_model_if_exists

if MODEL_NAME in TRANSFER_MODELS:
    best_checkpoint_path = CHECKPOINT_DIR / f"{MODEL_NAME}_stage2" / "best_model.pth"
else:
    best_checkpoint_path = CHECKPOINT_DIR / MODEL_NAME / "best_model.pth"

selected_model = load_best_model_if_exists(
    model=selected_model,
    checkpoint_path=best_checkpoint_path,
    device=DEVICE,
)

predicted_numeric_labels = predict(selected_model, test_loader, device=DEVICE)
predicted_numeric_labels = predicted_numeric_labels.numpy()


Convert numeric predictions back to text labels.


In [ ]:
predicted_text_labels = label_encoder.inverse_transform(predicted_numeric_labels)


Load the sample submission format and create a final CSV with the required `Id` and `Expected` columns.


In [ ]:
from submission import create_submission

my_submission, sample_submission = create_submission(
    test_files=test_files,
    predicted_labels=predicted_text_labels,
    sample_submission_path=SAMPLE_SUBMISSION_PATH,
)

sample_submission.head(10)


In [ ]:
my_submission.head(10)


In [ ]:
# The CSV below can be uploaded to Kaggle or used for local review.


In [ ]:
my_submission.to_csv('simple_cnn_baseline.csv', index=False)
